# Information Extraction with SIMBA

This notebook adapts the DSPy facility support analyzer workflow to the `SIMBA` optimizer.

We build a structured information extraction pipeline for facility support messages and predict three outputs for each message:

1. **Urgency**: `low`, `medium`, or `high`
2. **Sentiment**: `positive`, `neutral`, or `negative`
3. **Categories**: the facility-management categories mentioned in the message

The notebook walks through:
- configuring the language model used by DSPy
- loading and splitting the facility-support dataset
- defining a multi-predictor DSPy program for the three extraction tasks
- measuring baseline performance on the held-out test set
- optimizing the program with `SIMBA`
- inspecting the updated predictor instructions and reevaluating the optimized program

Unlike the other optimizers in this folder, this notebook focuses on the `SIMBA` compilation flow shown in the code cells below.


In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



## Configure Language Model

The language model comes from `LLM_PROVIDER` in `.env`. `supportvectors` uses the SV cluster (`gpt-oss-20b`); any other value uses hosted OpenAI (`gpt-5-mini`).


In [2]:
# LLM_PROVIDER in .env selects the backend:
#   supportvectors -> SV cluster (gpt-oss-20b)
#   anything else  -> hosted OpenAI (gpt-5-mini; needs OPENAI_API_KEY)
import dspy
from svlearn_dspy.llm import make_lm

lm = make_lm()
dspy.configure(lm=lm)


## Load and Prepare Dataset

Download the facility support analyzer dataset from the `llama-prompt-ops` repository, convert each row into a `dspy.Example`, and prepare the records for training and evaluation.


In [3]:
import requests
import dspy
import json
import random

def init_dataset():
    # Load from the url
    url = "https://raw.githubusercontent.com/meta-llama/llama-prompt-ops/refs/heads/main/use-cases/facility-support-analyzer/dataset.json"
    dataset = json.loads(requests.get(url).text)
    dspy_dataset = [
        dspy.Example({
            "message": d['fields']['input'],
            "answer": d['answer'],
        }).with_inputs("message")
        for d in dataset
    ]
    random.Random(0).shuffle(dspy_dataset)
    train_set = dspy_dataset[:int(len(dspy_dataset) * 0.33)]
    val_set = dspy_dataset[int(len(dspy_dataset) * 0.33):int(len(dspy_dataset) * 0.66)]
    test_set = dspy_dataset[int(len(dspy_dataset) * 0.66):]

    return train_set, val_set, test_set

## Initialize Dataset Splits

Shuffle the dataset with a fixed random seed and create train, validation, and test splits. This cell also prints the size of each split so we can confirm the data was loaded correctly.


In [4]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(66, 66, 68)

## Examine Example Data

Preview one training example to see the raw facility-support message and the corresponding gold JSON-style answer that the program is expected to reproduce.


In [5]:
print("Input Message:")
print(train_set[0]['message'])

print("\n\nGold Answer:")
for k, v in json.loads(train_set[0]['answer']).items():
    print(f"{k}: {v}")

Input Message:
Subject: Adjusting Bi-Weekly Cleaning Schedule for My Office

Dear ProCare Facility Solutions Support Team,

I hope this message finds you well. My name is Dr. Alex Turner, and I have been utilizing your services for my office space for the past year. I must say, your team's dedication to maintaining a pristine environment has been commendable and greatly appreciated.

I am reaching out to discuss the scheduling of our regular cleaning services. While I find the logistical challenges of coordinating these services intellectually stimulating, I believe we could optimize the current schedule to better suit the needs of my team and our workflow. Specifically, I would like to explore the possibility of adjusting our cleaning schedule to a bi-weekly arrangement, ideally on Tuesdays and Fridays, to ensure our workspace remains consistently clean without disrupting our research activities.

Previously, I have attempted to adjust the schedule through the online portal, but I enc

## Define DSPy Module

Create the DSPy program for this task:
- three signatures define the urgency, sentiment, and category outputs
- a single module combines three `ChainOfThought` predictors
- each predictor reads the same message and produces one part of the final structured answer


In [6]:
from typing import List, Literal


class FacilitySupportAnalyzerUrgency(dspy.Signature):
    """
    Read the provided message and determine the urgency.
    """
    message: str = dspy.InputField()
    urgency: Literal['low', 'medium', 'high'] = dspy.OutputField()

class FacilitySupportAnalyzerSentiment(dspy.Signature):
    """
    Read the provided message and determine the sentiment.
    """
    message: str = dspy.InputField()
    sentiment: Literal['positive', 'neutral', 'negative'] = dspy.OutputField()

class FacilitySupportAnalyzerCategories(dspy.Signature):
    """
    Read the provided message and determine the set of categories applicable to the message.
    """
    message: str = dspy.InputField()
    categories: List[Literal["emergency_repair_services", "routine_maintenance_requests", "quality_and_safety_concerns", "specialized_cleaning_services", "general_inquiries", "sustainability_and_environmental_practices", "training_and_support_requests", "cleaning_services_scheduling", "customer_feedback_and_complaints", "facility_management_issues"]] = dspy.OutputField()

class FacilitySupportAnalyzerMM(dspy.Module):
    def __init__(self):
        self.urgency_module = dspy.ChainOfThought(FacilitySupportAnalyzerUrgency)
        self.sentiment_module = dspy.ChainOfThought(FacilitySupportAnalyzerSentiment)
        self.categories_module = dspy.ChainOfThought(FacilitySupportAnalyzerCategories)
    
    def forward(self, message: str):
        urgency = self.urgency_module(message=message)
        sentiment = self.sentiment_module(message=message)
        categories = self.categories_module(message=message)

        return dspy.Prediction(
            urgency=urgency.urgency,
            sentiment=sentiment.sentiment,
            categories=categories.categories
        )

program = FacilitySupportAnalyzerMM()

## Define Evaluation Metrics

Implement per-task scoring for urgency, sentiment, and categories, then combine them into a single overall metric. This metric is used both for baseline evaluation and by the optimizer during compilation.


In [7]:
def score_urgency(gold_urgency, pred_urgency):
    """
    Compute score for the urgency module.
    """
    score = 1.0 if gold_urgency == pred_urgency else 0.0
    return score

def score_sentiment(gold_sentiment, pred_sentiment):
    """
    Compute score for the sentiment module.
    """
    score = 1.0 if gold_sentiment == pred_sentiment else 0.0
    return score

def score_categories(gold_categories, pred_categories):
    """
    Compute score for the categories module.
    Uses the same match/mismatch logic as category accuracy in the score.
    """
    correct = 0
    for k, v in gold_categories.items():
        if v and k in pred_categories:
            correct += 1
        elif not v and k not in pred_categories:
            correct += 1
    score = correct / len(gold_categories)
    return score

def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Computes a score based on agreement between prediction and gold standard for categories, sentiment, and urgency.
    Returns the score (float).
    """
    # Parse gold standard from example
    gold = json.loads(example['answer'])

    # Compute scores for all modules
    score_urgency_val = score_urgency(gold['urgency'], pred.urgency)
    score_sentiment_val = score_sentiment(gold['sentiment'], pred.sentiment)
    score_categories_val = score_categories(gold['categories'], pred.categories)

    # Overall score: average of the three accuracies
    total = (score_urgency_val + score_sentiment_val + score_categories_val) / 3

    return total

## Evaluate Baseline Performance

Run the unoptimized program on the test set to establish a baseline before applying `SIMBA`. This gives us a reference point for comparing the optimizer's impact.


In [8]:
import dspy
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 52.47 / 68 (77.2%): 100%|██████████| 68/68 [00:00<00:00, 73.80it/s]

2026/10/20 17:50:15 INFO dspy.evaluate.evaluate: Average Metric: 52.46666666666667 / 68 (77.2%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, tr...",✔️ [0.933]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, quality_and_safety_concerns, facili...",✔️ [0.933]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, facility_management_issues]",✔️ [0.633]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.633]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, cu...",✔️ [0.600]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,"[facility_management_issues, quality_and_safety_concerns]",✔️ [0.967]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[training_and_support_requests, general_inquiries]",✔️ [0.967]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,negative,"[routine_maintenance_requests, facility_management_issues, custome...",✔️ [0.667]


EvaluationResult(score=77.16, results=<list of 68 results>)

## Initialize SIMBA Optimizer

Instantiate `SIMBA` with the notebook's metric and thread configuration. This optimizer will search for stronger predictor instructions using the training set provided during compilation.


In [9]:
from dspy import SIMBA

optimizer = SIMBA(
    metric=metric,
    # Cheap validated budget (defaults are bsize=32, num_candidates=6,
    # max_steps=8 - roughly 4x the tokens). Raise these for stronger gains.
    bsize=16,
    num_candidates=4,
    max_steps=4,
    num_threads=16,
)

## Run SIMBA Optimization

Compile the program with `SIMBA` using the training split. This step produces an optimized version of the original program while leaving the program structure itself unchanged.


In [10]:
optimized_program = optimizer.compile(
    program,
    trainset=train_set,
)

2026/10/20 17:50:15 INFO dspy.teleprompt.simba: Starting batch 1 of 4.
2026/10/20 17:50:15 INFO dspy.teleprompt.simba: Sampling program trajectories on 16 examples x 4 samples.


Processed 64 / 64 examples: 100%|██████████| 64/64 [00:54<00:00,  1.18it/s]

2026/10/20 17:51:09 INFO dspy.teleprompt.simba: Batch 1: Baseline mini-batch score: 0.7416666666666667

2026/10/20 17:51:09 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #1, with max score 0.9333333333333332, max-to-min gap 0.3666666666666666, and max-to-avg gap 0.12499999999999989.
2026/10/20 17:51:09 INFO dspy.teleprompt.simba: Batch 1: Invoking strategy: append_a_demo_
2026/10/20 17:51:09 INFO dspy.teleprompt.simba_utils: Added 3 demos (one each) across all predictors.
2026/10/20 17:51:09 INFO dspy.teleprompt.simba: 

2026/10/20 17:51:09 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #2, with max score 0.9666666666666667, max-to-min gap 0.06666666666666665, and max-to-avg gap 0.033333333333333326.
2026/10/20 17:51:09 INFO dspy.teleprompt.simba: Batch 1: Invoking strategy: append_a_demo_
2026/10/20 17:51:09 INFO dspy.teleprompt.simba_utils: Added 3 demos (one each) across all predictors.
2026/10/20 17:51:09 INFO dspy.teleprompt.simba: 

2026/10/20 17:51:09 INFO dspy.

2026/10/20 17:51:32 INFO dspy.teleprompt.simba_utils: Advice for urgency_module.predict: If the message contains explicit urgency signals such as the words/phrases "urgent", "urgent inquiry", "high-priority", a direct request for a "prompt response", or describes recent unexpected problems that could cause operational disruption or downtime, then label urgency as 'high'. In your reasoning, list the concrete cues you used (e.g., quote "high-priority"; mention "prompt response" request; mention "unexpected maintenance issues" and risk of disruption). If the message lacks those markers and instead speaks about future planning or general questions without time pressure, label 'medium' or 'low' and explain which absence of cues led to that choice. Always prefer explicit textual cues and operational-impact language over inferred tone.
2026/10/20 17:51:32 INFO dspy.teleprompt.simba_utils: Advice for sentiment_module.predict: If the message is an initial business inquiry that is polite, factua

Processed 80 / 80 examples: 100%|██████████| 80/80 [01:23<00:00,  1.04s/it]

2026/10/20 17:52:55 INFO dspy.teleprompt.simba: Scores after 1 batches: [0.75, 0.6583333333333333, 0.6791666666666667, 0.7395833333333333, 0.73125], Best: 0.75

2026/10/20 17:52:55 INFO dspy.teleprompt.simba: Starting batch 2 of 4.
2026/10/20 17:52:55 INFO dspy.teleprompt.simba: Sampling program trajectories on 16 examples x 4 samples.


... (15 outputs left out)

Processed 80 / 80 examples: 100%|██████████| 80/80 [01:49<00:00,  1.36s/it]

2026/10/20 18:05:19 INFO dspy.teleprompt.simba: Scores after 4 batches: [0.8354166666666666, 0.7395833333333333, 0.7833333333333333, 0.8645833333333333, 0.7999999999999999], Best: 0.8645833333333333

2026/10/20 18:05:19 INFO dspy.teleprompt.simba: VALIDATION: Evaluating 5 programs on the full trainset.



Processed 330 / 330 examples: 100%|██████████| 330/330 [05:41<00:00,  1.03s/it]

2026/10/20 18:11:01 INFO dspy.teleprompt.simba: Final trainset scores: [0.7555555555555555, 0.7545454545454545, 0.7707070707070707, 0.8454545454545455, 0.8595959595959596], Best: 0.8595959595959596 (at index 4)





## Inspect Optimized Prompts

Print the instructions learned for each predictor after `SIMBA` finishes so we can see how the optimizer revised the prompts for urgency, sentiment, and category extraction.


In [11]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: urgency_module.predict
Prompt:
Read the provided message and determine the urgency.

If the message explicitly uses words like 'urgent' or 'ASAP' in the subject/body OR requests prompt in-person help for a core system (HVAC, heat/cooling, gas/electrical) OR expresses inability/confidence that could lead to damage if left unaddressed (e.g., 'worried I might mess something up', 'not confident I'm doing it right'), then set urgency to 'high' and include a short justification (explicit urgent request + risk of damage or core-system impact). If the message requests help but does not use urgent markers and there is no safety/failure language, set 'medium'. Reserve 'low' for non-urgent scheduling/planned queries (no urgent wording, no risk, no immediate assistance requested).

If the message reports an ongoing problem that the sender tried to fix (phrases like "I attempted", "I tried to tighten", "it persists", "has been persistent") AND the sender asks for a technician/visit or us

## Evaluate Optimized Program

Run the same evaluation procedure on the optimized program to compare its performance against the original baseline.


In [12]:
evaluate(optimized_program)

Average Metric: 59.07 / 68 (86.9%): 100%|██████████| 68/68 [00:49<00:00,  1.38it/s]

2026/10/20 18:11:50 INFO dspy.evaluate.evaluate: Average Metric: 59.06666666666666 / 68 (86.9%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, training_and_support_...",✔️ [0.967]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",high,neutral,[routine_maintenance_requests],✔️ [0.333]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",high,neutral,[routine_maintenance_requests],✔️ [0.667]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, training_and_support_...",✔️ [0.933]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,neutral,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,neutral,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,neutral,"[facility_management_issues, customer_feedback_and_complaints, qua...",✔️ [0.600]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[training_and_support_requests, general_inquiries]",✔️ [0.967]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",high,neutral,"[routine_maintenance_requests, quality_and_safety_concerns, custom...",✔️ [0.667]


EvaluationResult(score=86.86, results=<list of 68 results>)